In [1]:
import pandas as pd

movies = pd.read_csv('../data/movies.csv') # accessing movies
ratings = pd.read_csv('../data/ratings.csv') # accessing ratings

print(movies.head())
print(f"\nShape: {movies.shape}")

   movieId                               title  \
0        1                    Toy Story (1995)   
1        2                      Jumanji (1995)   
2        3             Grumpier Old Men (1995)   
3        4            Waiting to Exhale (1995)   
4        5  Father of the Bride Part II (1995)   

                                        genres  
0  Adventure|Animation|Children|Comedy|Fantasy  
1                   Adventure|Children|Fantasy  
2                               Comedy|Romance  
3                         Comedy|Drama|Romance  
4                                       Comedy  

Shape: (9742, 3)


In [5]:
from sklearn.feature_extraction.text import TfidfVectorizer

movies['genres_clean'] = movies['genres'].str.replace('|', ' ', regex = False) # since genres separated by |, we get rid of it and replace with space

tfidf = TfidfVectorizer(stop_words = 'english')
tfidf_matrix = tfidf.fit_transform(movies['genres_clean'])

print(f"Matrix shape: {tfidf_matrix.shape}") # the (9742, 23) is the 9742 movies, 23 genres


Matrix shape: (9742, 23)


In [6]:
from sklearn.metrics.pairwise import cosine_similarity
import numpy as np

cosine_sim = cosine_similarity(tfidf_matrix, tfidf_matrix)
print(f"Similarity matrix shape: {cosine_sim.shape}")

Similarity matrix shape: (9742, 9742)


In [8]:
indices = pd.Series(movies.index, index = movies['title']).drop_duplicates()

def get_recommendations(title, n = 10):
    if title not in indices:
        return f"Movie '{title}' not found."

    idx = indices[title]
    sim_scores = list(enumerate(cosine_sim[idx]))
    sim_scores = sorted(sim_scores, key=lambda x: x[1], reverse = True)
    sim_scores = sim_scores[1: n + 1] # since first movie is movie itself

    movie_indices = [i[0] for i in sim_scores]
    return movies['title'].iloc[movie_indices].tolist() # converts all of the movie titles into one big list

In [9]:
import joblib

joblib.dump(cosine_sim, '../data/cosine_sim.pkl')
movies.to_csv('../data/movies_clean.csv', index = False)

print("Model saved!")

Model saved!
